# Row 3: the cue matcher alone (stage 0 and 1a)

What this tests and why: `docs/RATIONALE.md` (section 4, row 3; decisions D17 to D19). Run the cells in order from the repo root.

1. The edge model turns each question into 3 visual cues (stage 0).
2. The embedding model compares every 2 s window, every 0.5 s, with those cues and we store the similarities (stage 1a).
3. On the CPU we try different trigger rules and thresholds on the stored numbers, and score them.

The embedding model uses about 4.4 GB of GPU memory, the decider about 4.3 GB: **they do not fit together on the 4060, so the
two stages run one after the other, as separate commands** (the cells below do this).

## 0. Setup and GPU check

In [4]:
import os
os.chdir(os.path.join(os.getcwd(), ".."))


In [2]:
%pip install -q -e ".[models]"
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(p.name, f"{p.total_memory/1e9:.1f} GB")

Note: you may need to restart the kernel to use updated packages.
torch 2.6.0+cu124 | cuda: True
NVIDIA GeForce RTX 4060 Laptop GPU 8.6 GB


ERROR: file:///C:/Users/abhin/Code/CV_Proj/notebooks does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


## 1. Videos (skip if the val videos are already downloaded)

In [ ]:
!python scripts/ovo_tar_remote.py fetch --samples data/processed/ovo_far_val.jsonl --out data/raw/ovo/src_videos

## 2. Stage 0: the cues

The edge model reads each question and writes 3 short descriptions of what the camera would see at the right moment.
Takes a few minutes for the val questions.

In [5]:
!python scripts/run_cues.py --samples data/processed/ovo_far_val.jsonl

cues -> results\cache\cues\Qwen3-VL-2B-Instruct__b055d726\cues.jsonl  (43 questions)
  ovo_CRR_1479 CRR: ['Woman steps through a red door, hand on the frame', 'Woman enters the center door, hand on the handle', 'Woman exits the red door, hand on the frame']
  ovo_CRR_1480 CRR: ['The man lifts a book with his right hand.', 'He places the book on a table beside him.', 'He picks up a second book with his left hand.']
  ovo_CRR_1481 CRR: ['The man turns his head toward the right side of the frame.', 'A hand reaches out from the left side of the frame.', 'A red object is visible on the ground in front of him.']
  ovo_CRR_1482 CRR: ['A man in a red shirt walks past a white bench.', 'A woman in a green dress stands near a flower bed.', 'A child in a blue jacket runs toward the man.']
  ovo_CRR_1483 CRR: ["The police car's rear window shows a clear view of the road ahead.", "A red emergency light flashes on the car's dashboard.", "The officer's hand grips the steering wheel tightly."]
  ovo_CR


Loading weights: 100%|██████████| 625/625 [00:02<00:00, 247.50it/s]


## 2b. Read the cues

Skim these: if the cues are vague or wrong, the matcher cannot work well, and that is worth knowing before waiting for embeddings.

In [6]:
import glob, json
f = sorted(glob.glob("results/cache/cues/*/cues.jsonl"))[-1]
rows = [json.loads(l) for l in open(f) if l.strip()]
print(f, len(rows), "questions")
for r in rows[:12]:
    print(f"\n{r['id']}: {r['query']}")
    for c in r["cues"]: print("   -", c)

results/cache/cues\Qwen3-VL-2B-Instruct__b055d726\cues.jsonl 43 questions

ovo_CRR_1479: Women came out of different colors of doors and went into the center door. What is they purpose of doing so?
   - Woman steps through a red door, hand on the frame
   - Woman enters the center door, hand on the handle
   - Woman exits the red door, hand on the frame

ovo_CRR_1480: The man picked up several books from the ground. What does the man do with the books he picked up?
   - The man lifts a book with his right hand.
   - He places the book on a table beside him.
   - He picks up a second book with his left hand.

ovo_CRR_1481: The man whose suit is has deeper color turn his head and look at some, what is he looking at?
   - The man turns his head toward the right side of the frame.
   - A hand reaches out from the left side of the frame.
   - A red object is visible on the ground in front of him.

ovo_CRR_1482: The man is walking in the garden. Who does he come across?
   - A man in a red s

## 3. Stage 1a smoke test: two of the shortest val questions

In [7]:
import glob
CUES = sorted(glob.glob("results/cache/cues/*/cues.jsonl"))[-1]
!python scripts/run_embed_cache.py --samples data/processed/ovo_far_val.jsonl --cues {CUES} --limit 2 --shortest

cache: results\cache\matcher\Ops-MM-embedding-v1-2B__875248e9  (2 questions with cues)
[1/2] ovo_REC_1566 REC: ran, 38 steps, median 719 ms/step
[2/2] ovo_REC_1591 REC: ran, 42 steps, median 721 ms/step
done in 60 s


c:\Users\abhin\Code\CV_Proj\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\abhin\.cache\huggingface\hub\models--OpenSearch-AI--Ops-MM-embedding-v1-2B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
c:\Users\abhin\Code\CV_Proj\.venv

## 4. Speed check

Median milliseconds per matcher step, and what that means for the whole val split (about 7,900 steps).

In [8]:
import glob, json, statistics
files = glob.glob("results/cache/matcher/*/ovo_*.json")
ms = [json.load(open(f))["ms_median"] for f in files if json.load(open(f)).get("ms_median")]
print(f"{len(files)} questions stored; median {statistics.median(ms):.0f} ms per step")
print(f"projected for val (about 7,900 steps): {statistics.median(ms)*7900/60000:.0f} minutes")

2 questions stored; median 720 ms per step
projected for val (about 7,900 steps): 95 minutes


## 5. Stage 1a on all val questions (resumable: if it stops, run this cell again)

In [9]:
!python scripts/run_embed_cache.py --samples data/processed/ovo_far_val.jsonl --cues {CUES}

cache: results\cache\matcher\Ops-MM-embedding-v1-2B__875248e9  (43 questions with cues)
[1/43] ovo_CRR_1479 CRR: ran, 180 steps, median 727 ms/step
[2/43] ovo_CRR_1480 CRR: ran, 80 steps, median 773 ms/step
[3/43] ovo_CRR_1481 CRR: ran, 72 steps, median 786 ms/step
[4/43] ovo_CRR_1482 CRR: ran, 68 steps, median 795 ms/step
[5/43] ovo_CRR_1483 CRR: ran, 206 steps, median 802 ms/step
[6/43] ovo_CRR_1484 CRR: ran, 80 steps, median 811 ms/step
[7/43] ovo_CRR_1485 CRR: ran, 74 steps, median 813 ms/step
[8/43] ovo_SSR_1518 SSR: ran, 262 steps, median 832 ms/step
[9/43] ovo_SSR_1519 SSR: ran, 260 steps, median 856 ms/step
[10/43] ovo_SSR_1531 SSR: ran, 256 steps, median 836 ms/step
[11/43] ovo_SSR_1533 SSR: ran, 316 steps, median 836 ms/step
[12/43] ovo_SSR_1538 SSR: ran, 1074 steps, median 836 ms/step
[13/43] ovo_SSR_1540 SSR: ran, 316 steps, median 844 ms/step
[14/43] ovo_SSR_1545 SSR: ran, 384 steps, median 882 ms/step
[15/43] ovo_SSR_1546 SSR: ran, 136 steps, median 857 ms/step
[16/43] ov

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 729/729 [00:01<00:00, 418.55it/s]
c:\Users\abhin\Code\CV_Proj\.venv\Lib\site-packages\transformers\models\qwen2_vl\image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


## 6. Score the matcher alone

Prints the Gate 2 diagnostic (does the similarity separate moments near an event from the rest?), then a table of
average F1 for each trigger rule, cue combination, threshold and duplicate handling.

In [ ]:
import glob
cache = sorted(glob.glob("results/cache/matcher/Ops-MM-embedding-v1-2B__*"))[-1]
!python scripts/replay_matcher.py --cache {cache} --samples data/processed/ovo_far_val.jsonl

## 7. Push the results

The `.json` files (similarities) and the cues are small and get pushed; the `.npy` window vectors are large, are kept
local on purpose and are ignored by git. In a terminal:
```
git add results/cache && git commit -m "Row 3 cues and matcher similarities, val" && git pull --rebase origin abhi && git push origin abhi
```